# Camera-ready LRao — definitive run (all scenes, 5 seeds)
**Upload-and-run on a GPU runtime** (CPU also works — minutes per seed).

Trains the fixed LRao on the three spatial scenes (**pavia4, sandiego,
sandiego2**), **5 seeds each**, and scores the full amplitude sweep
(θ = 0.03 … 0.95, the same grid and the same planted test sets as the
archived detector sweep — labels match npz-for-npz).

**Recipe** = the archived July-7 generality run (Pavia AUC 0.744), recovered
from its transcript — robust median/MAD normalization inside the net,
`sigma_cutoff=1e-22` in the LFI loss, Adam 5e-4 / wd 5e-5, batch 2048,
≤1000 epochs with early stopping (patience 10, min_delta 1e-3) — with ONE
change: **hidden_dims=[128], the same one-hidden-layer architecture as DART**.

**Archived per run**: a checkpoint every 10 epochs + best/final weights +
loss history; raw train-box and test scores per (scene, seed, θ); per-seed
metrics (AUC, pAUC, Pd@0.05, Pd_cfar, Pfa; per-class Pfa on Pavia).
Every cell is resumable — rerunning skips finished seeds.

At the end: one zip (`lrao_camera_ready.zip`) downloads automatically.

In [ ]:
!git clone -b tsp-repro --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import sys, os, torch
sys.path.insert(0, '.')
import tsp_repro
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
for p in ('colab_deep/data/pavia-u.mat', 'tsp_repro/data/Sandiego.mat',
          'tsp_repro/data/Sandiego2.mat', 'tsp_repro/data/sandiego_regions.json',
          'tsp_repro/data/sandiego2_regions.json', 'tsp_repro/configs/spatial.yaml'):
    assert os.path.exists(p), f'missing {p}'
print('all bundled datasets present')

In [ ]:
from tsp_repro import lrao_camera_ready as LC
print('recipe:'); [print(f'  {k}: {v}') for k, v in LC.RECIPE.items()]
OUT = 'results/lrao_camera_ready'

## Train + sweep, one cell per scene (each resumable)

In [ ]:
LC.run_scene('pavia4', out_root=OUT, device=DEVICE)

In [ ]:
LC.run_scene('sandiego', out_root=OUT, device=DEVICE)

In [ ]:
LC.run_scene('sandiego2', out_root=OUT, device=DEVICE)

## Summary — AUC grid per scene + the Table row at θ=0.15

In [ ]:
LC.summarize(OUT)

## Zip + download
`ckpt/` (every-10-epoch checkpoints, best/final, histories),
`scores/` (raw train/test scores incl. labels), `metrics__*.json`.

In [ ]:
LC.make_zip(OUT, 'lrao_camera_ready.zip')